# NOTEBOOK FEATURE ENGINEERING

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import chi2_contingency
from itertools import combinations
from scipy.stats import f_oneway

import os
import re

from IPython.display import display, Markdown

import missingno as msno
import sys

from rapidfuzz import process, fuzz

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from itertools import product

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"..\00_Data\00_Processed\df_eda.csv")

In [3]:
df.columns

Index(['sex', 'race', 'age_cat', 'decile_score', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'score_text', 'screening_date', 'two_year_recid',
       'total_priors_count', 'c_charge_degree', 'c_charge_desc', 'start',
       'end', 'event', 'juv_fel_count', 'juv_misd_count', 'juv_other_count',
       'person_id', 'agency_text', 'maritalstatus', 'language', 'rawscore',
       'age', 'juv_priors_count', 'adult_priors_count', 'is_violent',
       'is_drug', 'is_property', 'is_fraud', 'is_weapon', 'is_traffic',
       'is_sexual', 'is_public_disorder', 'is_justice_related', 'no_charge'],
      dtype='object')

In [4]:
lista_variables_modelo = [
    'person_id',
    'decile_score',
    'rawscore',
    'v_decile_score',
    'is_recid',
    'is_violent_recid',
    'two_year_recid',
    'sex',
    'race',
    'age',
    'age_cat',
    'total_priors_count',
    'adult_priors_count',
    'juv_priors_count',
    'c_charge_degree',
    'maritalstatus',
    'juv_fel_count',
    'juv_misd_count',
    'juv_other_count',
    'is_violent',
    'is_drug',
    'is_property',
    'is_fraud',
    'is_weapon',
    'is_traffic',
    'is_sexual',
    'is_public_disorder',
    'is_justice_related',
    'no_charge'
]

In [5]:
df = df[lista_variables_modelo]

In [6]:
df.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age', 'age_cat',
       'total_priors_count', 'adult_priors_count', 'juv_priors_count',
       'c_charge_degree', 'maritalstatus', 'juv_fel_count', 'juv_misd_count',
       'juv_other_count', 'is_violent', 'is_drug', 'is_property', 'is_fraud',
       'is_weapon', 'is_traffic', 'is_sexual', 'is_public_disorder',
       'is_justice_related', 'no_charge'],
      dtype='object')

In [7]:
df['race'] = df['race'].replace({'asian': 'other', 'native american': 'other'})

In [8]:
df.race.value_counts()

race
african-american    3085
caucasian           2093
hispanic             557
other                379
Name: count, dtype: int64

In [9]:
df["two_year_recid"].sum()

2209

In [10]:
df.shape

(6114, 29)

In [ ]:
df['sex'] = df['sex'].replace({'male': 0, 'female': 1})

In [12]:
df.sex.value_counts()

sex
0    4925
1    1189
Name: count, dtype: int64

In [13]:
df['c_charge_degree'] = df['c_charge_degree'].replace({'felony': 0, 'misdemeanor': 1})

In [14]:
df.c_charge_degree.value_counts()

c_charge_degree
0    3878
1    2236
Name: count, dtype: int64

In [15]:
df['maritalstatus'] = df['maritalstatus'].replace({'married': 'significant other', 'divorced': 'separated', 'widowed': 'other', 'unknown': 'other'})

In [16]:
df.maritalstatus.value_counts()

maritalstatus
single               4716
significant other     936
separated             408
other                  54
Name: count, dtype: int64

In [17]:
def one_hot_encoding(df, column, drop_val):
    encoder = OneHotEncoder(
    drop=[drop_val], 
    sparse_output=False
    )

    encoded = encoder.fit_transform(df[[column]])

    df_encoded = pd.DataFrame(
    encoded,
    columns = encoder.get_feature_names_out([column])
    )

    return df_encoded

In [18]:
df_model = pd.concat([df, one_hot_encoding(df, 'maritalstatus', 'single')], axis = 1)

In [19]:
df_model["log_juv_priors_count"] = np.log1p(df_model["juv_priors_count"])
df_model["log_adult_priors_count"] = np.log1p(df_model["adult_priors_count"])

In [20]:
#df_no_caucasian = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'caucasian')], axis = 1)

In [21]:
#df_model = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'african-american')], axis = 1)

In [22]:
df_model['adult_priors_freq']=(df_model['adult_priors_count']/(df_model['age']))


In [23]:
df_model['adult_priors_freq_2']=((df_model['adult_priors_count']+(df_model['total_priors_count'].mean()))/((df_model['age'])+(df_model['age']).mean()))

In [24]:
df_model['adult_priors_freq_2']=((df_model['adult_priors_count']+(df_model['adult_priors_count'].mean()))/((df_model['age'])+(df_model['age']).mean()))

In [25]:
tasa_media = df_model["adult_priors_count"].sum()/(df_model['age']).sum()

In [26]:
tasa_media

0.09264151891104526

In [27]:
beta = 5

In [28]:
alfa = tasa_media * beta

In [29]:
df_model['adult_priors_freq_3']=((df_model['adult_priors_count']+ alfa)/((df_model['age'])+ beta))


In [30]:
df_model.head()

,person_id,decile_score,rawscore,v_decile_score,is_recid,is_violent_recid,two_year_recid,sex,race,age,...,is_justice_related,no_charge,maritalstatus_other,maritalstatus_separated,maritalstatus_significant other,log_juv_priors_count,log_adult_priors_count,adult_priors_freq,adult_priors_freq_2,adult_priors_freq_3
0,62384.0,2,-3.03,1,1,0,1,0,hispanic,94,...,0,0,1.0,0.0,0.0,0.0,1.098612,0.021277,0.039638,0.024881
1,56279.0,1,-4.08,1,0,0,0,0,caucasian,80,...,0,1,0.0,1.0,0.0,0.0,1.609438,0.050000,0.062336,0.052508
2,50959.0,1,-4.50,1,0,0,0,0,hispanic,79,...,0,0,0.0,0.0,1.0,0.0,0.000000,0.000000,0.027040,0.005514
3,53038.0,1,-4.63,1,0,0,0,0,caucasian,77,...,0,0,0.0,0.0,1.0,0.0,0.000000,0.000000,0.027534,0.005649
4,56006.0,1,-4.05,1,0,0,0,0,caucasian,76,...,0,0,0.0,1.0,0.0,0.0,0.693147,0.013158,0.036999,0.018064


In [31]:
P_A = df_model["race"].value_counts(normalize=True)
P_Y = df_model["two_year_recid"].value_counts(normalize=True)
P_AY = df_model.groupby(["race","two_year_recid"]).size() / len(df_model)

#calcular pesos
def compute_weight(row):
    return (P_A[row["race"]] * P_Y[row["two_year_recid"]]) / P_AY[row["race"], row["two_year_recid"]]

df_model["weight"] = df_model.apply(compute_weight, axis=1)

In [32]:
df_model.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age', 'age_cat',
       'total_priors_count', 'adult_priors_count', 'juv_priors_count',
       'c_charge_degree', 'maritalstatus', 'juv_fel_count', 'juv_misd_count',
       'juv_other_count', 'is_violent', 'is_drug', 'is_property', 'is_fraud',
       'is_weapon', 'is_traffic', 'is_sexual', 'is_public_disorder',
       'is_justice_related', 'no_charge', 'maritalstatus_other',
       'maritalstatus_separated', 'maritalstatus_significant other',
       'log_juv_priors_count', 'log_adult_priors_count', 'adult_priors_freq',
       'adult_priors_freq_2', 'adult_priors_freq_3', 'weight'],
      dtype='object')

In [33]:
df_model.to_csv(r'..\00_Data\00_Processed\df_modelo.csv', index=False)